In [ ]:
def extract_rusun_data_final(narasi_text):
    """
    Fungsi ekstraksi final yang menangani semua kasus termasuk kasus khusus
    """
    # Mapping bulan
    bulan_map = {
        'JAN': 'Januari', 'JANUARI': 'Januari',
        'FEB': 'Februari', 'FEBRUARI': 'Februari', 
        'MAR': 'Maret', 'MARET': 'Maret',
        'APR': 'April', 'APRIL': 'April',
        'MEI': 'Mei',
        'JUN': 'Juni', 'JUNI': 'Juni',
        'JUL': 'Juli', 'JULI': 'Juli',
        'AGU': 'Agustus', 'AGUSTUS': 'Agustus',
        'SEP': 'September', 'SEPTEMBER': 'September',
        'OKT': 'Oktober', 'OKTOBER': 'Oktober',
        'NOV': 'November', 'NOVEMBER': 'November',
        'DES': 'Desember', 'DESEMBER': 'Desember'
    }
    
    result = {
        'nama': None,
        'kode_hunian': None,
        'nomor_hunian': None,
        'bulan': None,
        'tahun': None,
        'original_text': narasi_text,
        'extraction_method': 'none'
    }
    
    # Filter data yang tidak relevan
    skip_patterns = [
        r'F297',
        r'PELIMP',
        r'POTONGAN SEWA',
        r'TRF DARI',
        r'^[A-Z]\d+/'  # Pattern seperti F297/
    ]
    
    for pattern in skip_patterns:
        if re.search(pattern, narasi_text, re.IGNORECASE):
            result['extraction_method'] = 'skipped_non_rusun'
            return result
    
    # Skip jika tidak mengandung SETOR TUNAI
    if 'SETOR TUNAI' not in narasi_text:
        result['extraction_method'] = 'no_setor_tunai'
        return result
    
    # Deteksi dan handle data duplikasi
    if narasi_text.count('/') > 6:  # Banyak slash mengindikasikan duplikasi
        # Cari pola duplikasi dan ambil bagian pertama
        parts = narasi_text.split('RUSUN/')
        if len(parts) > 1:
            # Ambil bagian sebelum duplikasi pertama
            clean_part = 'SETOR TUNAI/' + parts[0].split('SETOR TUNAI/')[-1] + 'RUSUN/' + parts[1].split(' RUSUN/')[0]
            narasi_text = clean_part
            result['extraction_method'] = 'cleaned_duplication'
    
    # Pattern nama yang diperbaiki untuk menangani semua kasus
    nama_patterns = [
        # Pattern standard
        r'SETOR TUNAI/([^/]*?)(?:\s+(?:RUSUN|RSN|QQ)\s*[\d/]|\s+\d{8}|\s+[A-Z]\s*[IVX]*\s*\d+|$)',
        # Pattern untuk nama dengan duplikasi
        r'SETOR TUNAI/([A-Z\s]+?)(?:\s+[A-Z\s]+/\d{8})',
        # Pattern untuk nama yang diikuti langsung kode
        r'SETOR TUNAI/([A-Z\s]+?)\s*(\d{8})',
        # Pattern untuk kasus nama kompleks
        r'SETOR TUNAI/([A-Z][A-Z\s]*?)(?:\s+(?:PBY|AN|QQ))',
        # Pattern untuk kasus khusus seperti SALAMUN SEWA BLN
        r'SETOR TUNAI/([A-Z\s]+?)(?:\s+SEWA\s+BLN|\s+BLN|\s+ID\s*\d{8})',
        # Pattern untuk nama sebelum SEWA atau ID
        r'SETOR TUNAI/([A-Z][A-Z\s]*?)\s+(?:SEWA|ID)'
    ]
    
    for i, pattern in enumerate(nama_patterns):
        nama_match = re.search(pattern, narasi_text)
        if nama_match:
            nama_raw = nama_match.group(1).strip()
            # Bersihkan nama dari kata-kata yang tidak perlu
            nama_clean = re.sub(r'\b(RUSUN|RSN|QQ|AN|PBY|SEWA)\b', '', nama_raw).strip()
            # Bersihkan karakter khusus dan angka di akhir
            nama_clean = re.sub(r'\s*\d+\s*$', '', nama_clean).strip()
            # Ambil maksimal 3 kata pertama
            nama_parts = nama_clean.split()
            if nama_parts and len(' '.join(nama_parts[:3])) > 2:
                result['nama'] = ' '.join(nama_parts[:3])
                result['extraction_method'] = f'nama_pattern_{i+1}'
                break
    
    # Ekstrak kode hunian dengan pattern yang diperbaiki
    kode_patterns = [
        r'\b(\d{8})\b',                    # Standard 8 digit
        r'/(\d{8})/',                      # Dalam slash
        r'RSN(\d{8})',                     # Setelah RSN
        r'RUSUN/(\d{8})',                  # Setelah RUSUN/
        r'ID[:\s]*(\d{8})',                # Setelah ID (untuk kasus SALAMUN)
        r'/ID\s+(\d{8})',                  # /ID 02040210
    ]
    
    for pattern in kode_patterns:
        kode_match = re.search(pattern, narasi_text)
        if kode_match:
            result['kode_hunian'] = kode_match.group(1)
            break
    
    # Ekstrak nomor hunian dengan prioritas setelah kode hunian
    search_text = narasi_text
    if result['kode_hunian']:
        kode_pos = narasi_text.find(result['kode_hunian'])
        if kode_pos != -1:
            search_text = narasi_text[kode_pos + 8:]
    
    hunian_patterns = [
        r'\b([A-Z]\s*[IVX]+\s*\d+)\b',     # A II 5, BII18, D II 10
        r'\b([A-Z]\s*\d+)\b',              # C16, D4
        r'\b([A-Z][IVX]+\d+)\b',           # AII5, BIII10
        r'/([A-Z]\s*[IVX]*\s*\d+)/',      # Dalam slash
        r'NO\.?\s*([A-Z]\s*[IVX]*\s*\d+)', # Setelah NO.
        r'/([A-Z]\s+[IVX]+\s+\d+)$'       # Di akhir string seperti /D II 10
    ]
    
    for pattern in hunian_patterns:
        hunian_matches = re.findall(pattern, search_text)
        if hunian_matches:
            result['nomor_hunian'] = hunian_matches[0].replace('  ', ' ').strip()
            break
    
    # Ekstrak bulan dan tahun dengan pattern yang diperbaiki
    bulan_patterns = [
        r'\b(JANUARI|FEBRUARI|MARET|APRIL|MEI|JUNI|JULI|AGUSTUS|SEPTEMBER|OKTOBER|NOVEMBER|DESEMBER)\s*(\d{4})?\b',
        r'\b(JAN|FEB|MAR|APR|MEI|JUN|JUL|AGU|SEP|OKT|NOV|DES)(\d{2,4})?\b',
        r'BLN\s+(JANUARI|FEBRUARI|MARET|APRIL|MEI|JUNI|JULI|AGUSTUS|SEPTEMBER|OKTOBER|NOVEMBER|DESEMBER)\s*(\d{4})?',
        r'BLN\s+(JAN|FEB|MAR|APR|MEI|JUN|JUL|AGU|SEP|OKT|NOV|DES)\s*(\d{2,4})?',  # Untuk kasus BLN MEI 2025
        r'/(\w{3,4})\s*(\d{2,4})',         # Format /JUN25
        r'SEWA\s+BLN\s+(JANUARI|FEBRUARI|MARET|APRIL|MEI|JUNI|JULI|AGUSTUS|SEPTEMBER|OKTOBER|NOVEMBER|DESEMBER)\s*(\d{4})?'  # SEWA BLN MEI 2025
    ]
    
    for pattern in bulan_patterns:
        bulan_matches = re.findall(pattern, narasi_text, re.IGNORECASE)
        if bulan_matches:
            bulan_raw, tahun_raw = bulan_matches[-1]
            result['bulan'] = bulan_map.get(bulan_raw.upper(), bulan_raw.title())
            
            if tahun_raw:
                if len(tahun_raw) == 2:
                    tahun_int = int(tahun_raw)
                    if tahun_int <= 30:
                        result['tahun'] = str(2000 + tahun_int)
                    else:
                        result['tahun'] = str(1900 + tahun_int)
                else:
                    result['tahun'] = tahun_raw
            else:
                result['tahun'] = '2025'
            break
    
    # Set extraction method jika berhasil
    if result['nama'] and result['extraction_method'] == 'none':
        result['extraction_method'] = 'final_standard'
    
    return result